## SPARK: OVERVIEW
- Apache Spark is a distributed engine for large-scale data processing
- Spark DataFrames are the main abstraction used for structured data
- It supports SQL, batch processing, streaming, and machine learning
- Core documentation: https://spark.apache.org/docs/latest/
- PySpark API reference: https://spark.apache.org/docs/latest/api/python/

## Spark DataFrame basics
Spark usa DataFrame come struttura principale per dati strutturati distribuiti.
Per una collezione monodimensionale etichettata si usa un DataFrame a singola colonna.
Le trasformazioni sono ottimizzate dal motore Catalyst.
Pattern base di creazione: `spark.createDataFrame(data, schema=...)`

In [27]:
# connect
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import Row
import os
import tempfile
import urllib.request
from pathlib import Path

WORK_TMP_DIR = str(Path('.spark-tmp').resolve())
os.makedirs(WORK_TMP_DIR, exist_ok=True)

def create_spark_session(
    app_name: str = "DiffAPIspark",
    master_url: str = "local[*]",
    driver_bind_address: str = "0.0.0.0",
) -> "SparkSession":

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master(master_url)
        .config("spark.driver.bindAddress", driver_bind_address)
        .config("spark.local.dir", WORK_TMP_DIR)
        .config("spark.driver.extraJavaOptions", f"-Djava.io.tmpdir={WORK_TMP_DIR}")
        .getOrCreate()
    )
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    return spark

def read_csv_any(path: str):
    if path.startswith('http://') or path.startswith('https://'):
        local_path = os.path.join(WORK_TMP_DIR, os.path.basename(path))
        urllib.request.urlretrieve(path, local_path)
        path = local_path
    df = spark.read.option('header', True).option('inferSchema', True).csv(path)
    # If CSV header had an empty first column name, rename it to 'row_id' to avoid warnings
    cols = df.columns
    if len(cols) > 0 and (cols[0] == '' or cols[0] is None):
        df = df.withColumnRenamed(cols[0], 'row_id')
    # Also normalize common generated name _c0 to 'row_id' for consistency
    if 'row_id' not in df.columns and '_c0' in df.columns:
        df = df.withColumnRenamed('_c0', 'row_id')
    return df

In [ ]:
# Creating a one-column Spark DataFrame
spark = create_spark_session('One column DataFrame')
data = [('b', 1), ('a', 0), ('c', 2)]
s = spark.createDataFrame(data, ['key', 'value'])
s.show()

In [ ]:
s = spark.createDataFrame([('A', 5.0), ('B', 6.0), ('C', 12.0), ('D', 1.3)], ['index', 'serie_0'])
s.show()


In [ ]:
# series access
s.limit(2).show()

## Spark DataFrame
- I DataFrame Spark sono tabelle distribuite gestite dal cluster
- Sono l'astrazione principale per i dati strutturati in PySpark
- Un DataFrame a singola colonna copre i casi monodimensionali
- Creazione tipica: `spark.createDataFrame(data, schema)`

In [ ]:
spark = create_spark_session("Key-tuple DataFrame")

spark.createDataFrame([('A', 0.0), ('B', 4.0), ('C', 8.0), ('D', -12.0), ('E', 16.0)], ['key', 'foo']).show()

Hands-on excercise (slide 29)

In [ ]:
spark = create_spark_session("Tuple List DataFrame")

spark.createDataFrame([(1, 3), (2, 4), (6, 5)], ['foo', 'fee']).show()

In [ ]:
spark = create_spark_session("None Values DataFrame")

spark.createDataFrame([('a', 1, None), ('b', 1, None), ('c', None, 1), ('d', None, 2)], ['idx', 'foo', 'fee']).show()

## Spark DataFrame schema
Spark espone uno schema tipizzato con nomi colonna, tipi e nullability.
Usa `printSchema()` per l'ispezione strutturale.
Usa `select`, `filter`, `withColumn` e `groupBy` per trasformare i dati.

## Spark data types
- `IntegerType` and `LongType` for integers
- `FloatType`, `DoubleType`, and `DecimalType` for numeric values
- `BooleanType` for booleans
- `DateType` and `TimestampType` for temporal data
- `StringType` for text
- `ArrayType`, `MapType`, and `StructType` for nested structures

- `byte`, `short`, `integer`, `long`: signed integral types
- `float`, `double`, `decimal`: numeric types with fractional support
- `boolean`: logical values
- `date`, `timestamp`: date and time types
- `string`: textual data
- `binary`: raw bytes
- `array`, `map`, `struct`: nested types for complex data

### Spark DataFrame attributes
- `df.columns` lists column names
- `df.dtypes` returns the Spark type of each column
- `df.schema` returns the full schema object
- `df.columns`, `df.count()`, and `df.printSchema()` are the most common inspection tools
- `df.rdd` exposes the underlying RDD when lower-level access is needed

### Spark DataFrame methods
- `show(n)`, `limit(n)`: preview rows
- `describe()`, `summary()`: descriptive statistics
- `select()`, `filter()`, `where()`: project and filter data
- `dropna()`, `fillna()`: handle missing values
- `distinct()`, `dropDuplicates()`: remove duplicates
- `groupBy()`, `agg()`, `count()`: aggregate data

In [ ]:
# Ispezione schema-driven con Spark
spark = create_spark_session('schema-driven inspection')

spark.range(1).show()

In [ ]:
from pyspark.sql import Row
from pyspark.sql import types as T
spark = create_spark_session()

data = [
    Row(Colonna1=1, Colonna2=5.1, Colonna3='2022-01-01', Colonna4='A', Colonna5=True),
    Row(Colonna1=2, Colonna2=6.2, Colonna3='2022-02-01', Colonna4='B', Colonna5=False),
    Row(Colonna1=3, Colonna2=7.3, Colonna3='2022-03-01', Colonna4='A', Colonna5=True),
    Row(Colonna1=4, Colonna2=8.4, Colonna3='2022-04-01', Colonna4='C', Colonna5=False),
]

df = spark.createDataFrame(data)
df.show()

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql import functions as F
spark = create_spark_session()

df = df.withColumn('Colonna1', F.col('Colonna1').cast('double'))
df.show()

In [ ]:
spark = create_spark_session()

df = df.withColumn('Colonna2', F.col('Colonna2').cast('int'))
df.show()

In [ ]:
df = df.withColumn('Colonna3', F.to_date('Colonna3'))
df.printSchema()
df.show()


In [ ]:
# 4. Convertire la Colonna3 da data a stringa (Spark)
df = df.withColumn('Colonna3', F.date_format(F.col('Colonna3'), 'yyyy-MM-dd'))
df.printSchema()
df.show()

In [ ]:
# 5. Colonna4 come stringa (Spark non usa dtype category)
df = df.withColumn('Colonna4', F.col('Colonna4').cast('string'))
df.printSchema()
df.show()

In [ ]:
df = df.withColumn('Colonna4', F.col('Colonna4').cast('string'))
df.printSchema()
df.show()

In [ ]:
# 7. Convertire piu colonne contemporaneamente
from pyspark.sql import functions as F

df = (df
    .withColumn('Colonna1', F.col('Colonna1').cast('double'))
    .withColumn('Colonna2', F.col('Colonna2').cast('int'))
    .withColumn('Colonna4', F.col('Colonna4').cast('string'))
)
df.printSchema()
df.show()

In [ ]:
df = df.withColumn('Colonna5', F.col('Colonna5').cast('int'))
df.show()

In [ ]:
# 9. Convertire la Colonna5 da intero a booleano
df = df.withColumn('Colonna5', F.col('Colonna5').cast('boolean'))
df.printSchema()
df.show()

In [ ]:
category_map = F.create_map(F.lit('A'), F.lit(0.0), F.lit('B'), F.lit(1.0), F.lit('C'), F.lit(2.0))
df_numeric = (df.withColumn('Colonna1', F.col('Colonna1').cast('double')).withColumn('Colonna2', F.col('Colonna2').cast('double')).withColumn('Colonna3', F.unix_timestamp('Colonna3', 'yyyy-MM-dd').cast('double')).withColumn('Colonna4', F.element_at(category_map, F.col('Colonna4'))).withColumn('Colonna5', F.col('Colonna5').cast('double')))
df_numeric.printSchema()
df_numeric.show()


# Processing tables with Spark

## Slicing in a DataFrame



### Column slicing
- Select one column by name: `df['Colonna1']` (Output is a Series)

In [ ]:
s = df['Colonna1']

- Select multiple columns by name: `df[['sex','salary']]` (Output is a DataFrame)

In [ ]:
df_ap = df[['Colonna2', 'Colonna3']]

### Row slicing
- Specify range using ":": `df[10:20]` (Selects rows by position)

In [ ]:
df_ap = (
    df.withColumn('_rn', F.row_number().over(Window.orderBy(F.monotonically_increasing_id())))
      .filter((F.col('_rn') >= 2) & (F.col('_rn') <= 3))
      .drop('_rn')
)
df_ap.show()

# Equivalenza con SQL

# Relational Algebra

In [ ]:
import sqlite3


conn = sqlite3.connect(":memory:") # filename in alternative
cursor = conn.cursor()

# Creazione delle tabelle
cursor.executescript('''
CREATE TABLE temps (
    City VARCHAR(50),
    Temperature INT,
    Humid VARCHAR(3)
);

CREATE TABLE other_temps (
    City VARCHAR(50),
    Temperature INT,
    Humid VARCHAR(3)
);

CREATE TABLE other_temps_2 (
    Temperature INT
);

CREATE TABLE countries (
    City VARCHAR(50),
    Country VARCHAR(50)
);

INSERT INTO temps VALUES ('San Diego', 76, 'No'), ('San Diego', 79, 'No'), 
                         ('San Diego', 88, 'Yes'), ('Torino', 32, 'Yes'),
                         ('Rome', 56, 'Yes'), ('Milan', 42, 'Yes');

INSERT INTO other_temps VALUES ('Los Angeles', 79, 'No'), 
                              ('San Diego', 76, 'No'), ('Miami', 88, 'Yes');

INSERT INTO other_temps_2 VALUES (76), (79), (88);

INSERT INTO countries VALUES ('Toronto', 'Canada'), ('Shanghai', 'China'),
                            ('San Diego', 'USA'), ('Rome', 'Italy'), ('Milan', 'Italy');
''')

conn.commit()

print("Database creato e popolato con successo.")




In [ ]:
temps = spark.createDataFrame([('San Diego', 76, 'No'), ('San Diego', 79, 'No'), ('San Diego', 88, 'Yes'), ('Torino', 32, 'Yes'), ('Rome', 56, 'Yes'), ('Milan', 42, 'Yes')], ['City', 'Temperature', 'Humid'])
other_temps = spark.createDataFrame([('Los Angeles', 79, 'No'), ('San Diego', 76, 'No'), ('Miami', 88, 'Yes')], ['City', 'Temperature', 'Humid'])
other_temps_2 = spark.createDataFrame([(76,), (79,), (88,)], ['Temperature'])
countries = spark.createDataFrame([('Toronto', 'Canada'), ('Shanghai', 'China'), ('San Diego', 'USA'), ('Rome', 'Italy'), ('Milan', 'Italy')], ['City', 'Country'])



## Projection ($Π$)

Used to project (keep) columns in a relation. Duplicates rows are dropped.

$$
\Pi_{(\text{City, Humid})}(\text{temps})
$$





In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT DISTINCT City, Humid FROM temps;
''')
cursor.fetchall()

In [ ]:
temps[['City', 'Humid']].drop_duplicates()

## Selection (σ)

Used to keep rows in a relation that satisfy certain conditions.


$$
\sigma_{(\text{Temperature} > 50)}(\text{temps})
$$

In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT * FROM temps WHERE Temperature > 50;
''')
cursor.fetchall()

In [ ]:
temps[temps['Temperature'] > 50]

## Union ($⋃$)

$$
temps ⋃ other\_temps
$$

In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT * FROM temps
UNION
SELECT * FROM other_temps;
''')
cursor.fetchall()

In [ ]:
temps.unionByName(other_temps).dropDuplicates().show()


## Difference ($-$)

Used to find the rows that are in one relation but not the other. Only works if the two relations have the same attributes (column names).

$$
temps - other\_temps
$$

In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT * FROM temps
WHERE City NOT IN (SELECT City FROM other_temps);
''')
cursor.fetchall()

In [ ]:
other_city = other_temps.select('City').distinct()
temps.join(other_city, on='City', how='left_anti').show()

In [ ]:
from IPython.display import Image
Image(url='https://www.stationx.net/wp-content/uploads/2023/05/SQL-Joins.png',width=600,height=600)

## Inner Join (⨝)

Used to combine rows from two relations based on a common attribute. In our example, we join `temps` and `countries` on the column `City`.

$$
\text{temps} \bowtie \text{countries}
$$


|   | **Algoritmo Join** |
|---|-------------------------|
|   | **JOIN(temps, countries)** |
| 1 | Identifica l'attributo comune: `City` |
| 2 | Esegui il merge interno delle tabelle `temps` e `countries` sulla colonna `City` |
| 3 | Restituisci il risultato della join |


In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT t.*, c.Country 
FROM temps t
JOIN countries c ON t.City = c.City;
''')
cursor.fetchall()

In [ ]:
temps.join(countries, on='City', how='inner').show()

## Cross product ($×$)
$$
\text{temps} \times \text{countries}
$$

In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT * FROM temps CROSS JOIN countries;
''')
cursor.fetchall()

In [ ]:
temps.crossJoin(countries).show()


## Composed Operator example

$$
\Pi_{(\text{City, Humid})} \big(\sigma_{(\text{Temperature} > 50)}(\text{temps}) \big)
$$

In [ ]:
temps.filter(F.col('Temperature') > 50).select('City', 'Humid').dropDuplicates().show()

## Division (÷)

Finds all countries that have **all** cities whith all temperatures in in the `other_temps_2` table:



**Relazioni:**  
- \( R(City, Temperature) \): tabella `temps` (relazione principale)  
- \( S(Temperature) \): tabella `other_temps` (relazione dei valori da verificare)

**Formula:**  
La divisione si esprime con la formula:  

$$
R \div S = \pi_{City}(R) - \pi_{City}\left(\pi_{City}(R) \times S - R\right)
$$

**Significato:**  
1. $\pi_{City}(R)$: otteniamo l'insieme di tutte le città.
2. $\pi_{City}(R) \times S$: formiamo il prodotto cartesiano tra le città e le temperature richieste.
3. $\pi_{City}(R) \times S - R$: troviamo le città per cui manca almeno una delle temperature richieste.
4. Sottraendo tali città dall'insieme totale, rimangono solo quelle che hanno **tutte** le temperature in $S$.

|   | **Algoritmo Divisione** |
|---|-------------------------|
|   | **DIVISIONE(temps, other_temps)** |
| 1 | A = proiezione delle città in temps |
| 2 | B = proiezione delle temperature in other_temps |
| 3 | AxB = prodotto cartesiano A × B |
| 4 | matches = merge interno di AxB e R (dove R contiene (City, Temperature) di temps) |
| 5 | per ogni città in matches, contare le occorrenze; se il conteggio è compatibile con **B**, includerla nel risultato |
| 6 | Restituire le città che soddisfano la condizione |


In [ ]:
# Step 1: Proiettare le temperature richieste (S)
B = other_temps_2.select('Temperature').distinct()

# Step 2: Coppie (City, Temperature) presenti in temps (R)
R = temps.select('City', 'Temperature').distinct()

# Step 3: Trovare quante temperature richieste sono presenti per ogni città
required_count = B.count()
valid_cities = (R.join(B, on='Temperature', how='inner')
                .groupBy('City')
                .agg(F.countDistinct('Temperature').alias('matched_count'))
                .filter(F.col('matched_count') == required_count)
                .select('City'))

# Step 4: Risultato finale
valid_cities.show()

In [ ]:
cursor = conn.cursor()
cursor.execute('''
SELECT DISTINCT t1.City
FROM temps t1
WHERE NOT EXISTS (
    SELECT t2.Temperature 
    FROM other_temps_2 t2
    WHERE NOT EXISTS (
        SELECT * 
        FROM temps t3 
        WHERE t3.City = t1.City AND t3.Temperature = t2.Temperature
    )
);
''')
cursor.fetchall()

# Comparativa SQL Spark

### SELECT (Selezione delle colonne)

**SQL:**
```sql
SELECT Nome, Eta FROM tabella;
```

**Spark:**
```python
df.select('Nome', 'Eta')
```

### WHERE (Filtro sulle righe)

**SQL:**
```sql
SELECT * FROM tabella WHERE Eta > 30;
```

**Spark:**
```python
df.filter(F.col('Eta') > 30)
```

### SELECT + WHERE

**SQL:**
```sql
SELECT Nome, Punteggio FROM tabella WHERE Eta > 30 AND Sesso = 'M';
```

**Spark:**
```python
df.filter((F.col('Eta') > 30) & (F.col('Sesso') == 'M')).select('Nome', 'Punteggio')
```

### ORDER BY (Ordinamento)

**SQL:**
```sql
SELECT * FROM tabella ORDER BY Eta DESC;
```

**Spark:**
```python
df.orderBy(F.col('Eta').desc())
```

### GROUP BY (Raggruppamento)

**SQL:**
```sql
SELECT Sesso, AVG(Eta) FROM tabella GROUP BY Sesso;
```

**Spark:**
```python
df.groupBy('Sesso').agg(F.avg('Eta').alias('avg_Eta'))
```

### JOIN

**SQL:**
```sql
SELECT * FROM tabella1 JOIN tabella2 ON tabella1.ID = tabella2.ID;
```

**Spark:**
```python
df1.join(df2, on='ID', how='inner')
```

### LIMIT

**SQL:**
```sql
SELECT * FROM tabella LIMIT 3;
```

**Spark:**
```python
df.limit(3)
```

### DISTINCT

**SQL:**
```sql
SELECT DISTINCT Sesso FROM tabella;
```

**Spark:**
```python
df.select('Sesso').distinct()
```

----

# Esercizi Spark

In [ ]:
data = [('Alice', 25, 'F', 85), ('Bob', 30, 'M', 90), ('Charlie', 35, 'M', 88), ('David', 40, 'M', 92), ('Emma', 22, 'F', 78)]
df = spark.createDataFrame(data, ['Nome', 'Età', 'Sesso', 'Punteggio'])
df.show()


In [ ]:
# 1. Seleziona solo la colonna 'Nome'
nome_c = df.select('Nome')
nome_c.show()

In [ ]:
# 2. Seleziona le colonne 'Nome' e 'Età' (SELECT Nome, Età FROM tabella;)
nome_eta_c = df.select('Nome', 'Età')
nome_eta_c.show()

In [ ]:
# 3. Filtra le righe dove l'età è superiore a 30
df_filtrato = df.filter(F.col('Età') > 30) # (SELECT * FROM tabella WHERE Età > 30;)
df_filtrato.show()

In [ ]:
df_filtrato = df.filter((F.col('Età') > 30) & (F.col('Punteggio') > 90)) # (SELECT * FROM tabella WHERE 'Età' > 30 and 'Punteggio'>90;)
df_filtrato.show()

In [ ]:
df = df.withColumn('Voto', F.floor(F.rand(seed=42) * 41 + 60).cast('int'))
df.show()


In [ ]:
# 5. Ordina il DataFrame in base alla colonna 'Punteggio' in ordine decrescente
df_ordinato = df.orderBy(F.col('Punteggio').desc())
df_ordinato.show()

In [ ]:
# 6. Calcola la media dei punteggi
media = df.agg(F.avg('Punteggio').alias('media_punteggio')).collect()[0]['media_punteggio']
print(media)

In [ ]:
# 7. Trova la persona con il punteggio massimo
max_score = df.agg(F.max('Punteggio').alias('max_p')).collect()[0]['max_p']
punt = df.filter(F.col('Punteggio') == max_score)
punt.show()

In [ ]:
# 8. Cambia il sesso di 'Alice' a 'test'
df = df.withColumn('Sesso', F.when(F.col('Nome') == 'Alice', F.lit('test')).otherwise(F.col('Sesso')))
df.show()

In [ ]:
# Ripristina
df = df.withColumn('Sesso', F.when(F.col('Nome') == 'Alice', F.lit('F')).otherwise(F.col('Sesso')))
df.show()

In [ ]:
# 9. Conta il numero di persone per sesso
conteggio_sesso = df.groupBy('Sesso').count()
conteggio_sesso.show()

In [ ]:
df_rimosso = df.drop('Voto')
df_rimosso.show()


In [ ]:
# 11. Calcola la somma cumulativa della colonna 'Età'
print(df.agg(F.sum('Età').alias('somma_eta')).collect()[0]['somma_eta'])

In [ ]:
# 12. Trova le persone con punteggio superiore a 80 e età inferiore a 35
trovate = df.filter((F.col('Punteggio') > 80) & (F.col('Età') < 35))
trovate.show()

In [ ]:
# 13. Calcola la deviazione standard della colonna 'Punteggio'
print(df.agg(F.stddev('Punteggio').alias('std_punteggio')).collect()[0]['std_punteggio'])

In [ ]:
# 14. Raggruppa il DataFrame per sesso e calcola la media delle età
conteggio_sesso = df.groupBy('Sesso').agg(F.avg('Età').alias('eta_media'))
conteggio_sesso.show()

In [ ]:
# 15. Rinomina la colonna 'Nome' in 'Cognome'
df_renamed = df.withColumnRenamed('Nome', 'Cognome')
df_renamed.show()

In [ ]:
# Ripristina lo schema originale
df = df_renamed.withColumnRenamed('Cognome', 'Nome')
df.show()

In [ ]:
data2 = [('Alice', 'Ingegnere'), ('Bob', 'Analista'), ('Charlie', 'Manager'), ('David', 'Programmatore'), ('Emma', 'Designer')]
df2 = spark.createDataFrame(data2, ['Cognome', 'Lavoro'])
df2.show()


In [ ]:
from IPython.display import Image
Image(url='https://www.stationx.net/wp-content/uploads/2023/05/SQL-Joins.png',width=600,height=600)

# left join

In [ ]:
df_merged = df.join(df2, df.Nome == df2.Cognome, 'left')
df_merged.show()


In [ ]:
# togli colonna duplicata della join
df_merged = df_merged.drop('Cognome')
df_merged.show()

# inner join

In [ ]:
# Number of total cells
df.count() * len(df.columns)

In [ ]:
# 17. Salva il DataFrame in un file CSV
(df_merged
 .coalesce(1)
 .write
 .mode('overwrite')
 .option('header', True)
 .csv('06-dati.csv'))
print('CSV scritto in cartella 06-dati.csv')

In [ ]:
df_loaded = spark.read.option('header', True).option('inferSchema', True).csv('06-dati.csv')
df_loaded.show()


In [ ]:
nuova_riga = spark.createDataFrame([('Emma', 22, 'F', 99, 90, 'Imprenditore')], ['Nome', 'Età', 'Sesso', 'Punteggio', 'Voto', 'Lavoro'])
df_concat = df_loaded.unionByName(nuova_riga, allowMissingColumns=True)
df_concat.show()


## Reading data using Spark

```python
# Read csv file
df = spark.read.option('header', True).option('inferSchema', True).csv('myfile.csv')
```

```python
# Read xlsx file (richiede spark-excel package)
df = spark.read.format('com.crealytics.spark.excel').option('header', True).load('myfile.xlsx')
```

```python
# Read json file
df = spark.read.json('myfile.json')
```

In [ ]:
data = [('Alice', 'Alice Blues', 24), ('Bob', 'Bob Cartney', 56), ('Gian Franco', 'GianFranco Porri', 29), ('David', 'David Lee', 37)]
df = spark.createDataFrame(data, ['Nomi', 'Nomi_Cognomi', 'Anni'])
df.show(truncate=False)


In [ ]:
# Aggiungi colonne con lunghezza stringhe
df = (df
    .withColumn('Lunghezza_nomi', F.length(F.col('Nomi')))
    .withColumn('Lunghezza_n_c', F.length(F.col('Nomi_Cognomi')))
)
df.show(truncate=False)

In [ ]:
# ripristina
df = df.drop('Lunghezza_nomi', 'Lunghezza_n_c')
df.show(truncate=False)

In [ ]:
# Modifica tutte le stringhe in maiuscolo
df = df.withColumn('Nomi_Maiuscolo', F.upper(F.col('Nomi')))
df.show(truncate=False)

In [ ]:
df = df.withColumn('Lunghezza', F.length(F.col('Nomi')))
df.show(truncate=False)

In [ ]:
# Conta quante volte la lettera 'a' appare nella colonna Nomi
df = df.withColumn('Conta_A', F.length('Nomi') - F.length(F.regexp_replace('Nomi', 'a', '')))
df.show(truncate=False)

In [ ]:
# Estrai i primi tre caratteri
df = df.withColumn('Primi_3_char', F.substring(F.col('Nomi'), 1, 3))
df.show(truncate=False)

In [ ]:
# Estrai gli ultimi 2 caratteri
df = df.withColumn('Ultimi_2_char', F.expr('right(Nomi, 2)'))
df.show(truncate=False)

In [ ]:
# Sostituisci una sottostringa
df = df.withColumn('Nomi_mod', F.regexp_replace(F.col('Nomi'), 'a', 'X'))
df.show(truncate=False)

In [ ]:
# Colonna booleana: presenza di 'c'
df = df.withColumn('Contiene_C', F.instr(F.lower(F.col('Nomi')), 'c') > 0)
df.show(truncate=False)

In [ ]:
df = df.withColumn('Contiene_C', F.col('Contiene_C').cast('boolean'))
df.show(truncate=False)

In [ ]:
# Elimina spazi ai bordi
df = df.withColumn('Nomi_Senza_Spazi', F.trim(F.col('Nomi')))
df.show(truncate=False)

In [ ]:
df = df.withColumn('Nomi_Senza_Spazi', F.trim(F.col('Nomi')))
df.show(truncate=False)


In [ ]:
df = df.withColumn('Primo_Nome', F.split(F.col('Nomi_Senza_Spazi'), ' ').getItem(0))
df.show(truncate=False)


In [ ]:
df = df.withColumn('Cognome', F.element_at(F.split(F.col('Nomi_Cognomi'), ' '), -1))
df.show(truncate=False)


In [ ]:
df = df.withColumn('Conta_Parole', F.size(F.split(F.col('Nomi_Senza_Spazi'), ' ')))
df.show(truncate=False)


In [ ]:
df = df.withColumn('Nome_Completo', F.concat_ws(' ', F.col('Nomi_Senza_Spazi'), F.col('Cognome')))
df.show(truncate=False)


## date

In [ ]:
data = [(1, 5.1, '2022-01-01', 'A', True), (2, 6.2, '2022-02-01', 'B', False), (3, 7.3, '2022-03-01', 'A', True), (4, 8.4, '2022-04-01', 'C', False)]
df = spark.createDataFrame(data, ['Colonna1', 'Colonna2', 'Colonna3', 'Colonna4', 'Colonna5'])
df.show()


In [ ]:
df = df.withColumn('Colonna3', F.to_date('Colonna3'))
df.printSchema()
df.show()


In [ ]:
df = df.withColumn('Colonna3', F.date_format(F.col('Colonna3'), 'yyyy-MM-dd'))
df.printSchema()
df.show()


In [ ]:
data = [(1, 5.1, '2022-30-01', 'A', True), (2, 6.2, '2022-29-01', 'B', False), (3, 7.3, '2022-27-01', 'A', True), (4, 8.4, '2022-04-01', 'C', False)]
df = spark.createDataFrame(data, ['Colonna1', 'Colonna2', 'Colonna3', 'Colonna4', 'Colonna5'])
df.show()


In [ ]:
df = df.withColumn('Colonna3', F.to_date('Colonna3', 'yyyy-dd-MM'))
df.printSchema()
df.show()


# Salaries

### Common functions
`count`, `sum`, `mean`, `mad`, `median`, `min`, `max`, `mode`, `abs`, `prod`, `std`, `var`, `sem`, `skew`, `kurt`, `quantile`, `cumsum`, `cumprod`, `cummax`, `cummin`

In [ ]:
df = spark.read.option('header', True).option('inferSchema', True).csv('train-Salaries-DS.csv')
df.show(5)


In [ ]:
print(df.count())
print(df.select('*').count())


In [ ]:
# Number of total cells
df.count() * len(df.columns)

In [ ]:
df.columns


In [ ]:
df.printSchema()


In [ ]:
df.dtypes


In [ ]:
print((df.count(), len(df.columns)))


In [ ]:
df.describe().show()


In [ ]:
df.summary().show()


In [ ]:
df.groupBy('discipline').count().show()


## Filtering

- Use Spark boolean expressions with `filter` or `where`
- `df_sub = df.filter(F.col('salary') > 120000)`
- `df_f = df.filter(F.col('sex') == 'Female')`

### Operators
`>`, `>=`, `<`, `<=`, `==`, `!=`, `|` (or), `&` (and), `~` (not)

In [ ]:
df_app = df.groupBy('discipline').count().filter(F.col('count') > 1).select('discipline')
df_app.show()


In [ ]:
df_app.count()


In [ ]:
numeric_columns = ['phd', 'service', 'salary']


In [ ]:
df.select(*numeric_columns).describe().show()


In [ ]:
df.select(*numeric_columns).summary().show()


In [ ]:
df.limit(50).select(*numeric_columns).agg(*[F.mean(c).alias(c) for c in numeric_columns]).show()


## Slicing in a DataFrame

### Column slicing
- Select one column by name: `df['sex']` (Output is a Series)
- Select multiple columns by name: `df[['sex','salary']]` (Output is a DataFrame)

### Row slicing
- Specify range using ":": `df[10:20]` (Selects rows by position)

In [ ]:
print([name for name in dir(df) if name in ['select', 'filter', 'groupBy', 'agg', 'show', 'dropna', 'sample']])


In [ ]:
df.printSchema()


In [ ]:
df.orderBy(F.rand(seed=1)).limit(3).show()


In [ ]:
df.sample(0.1, seed=1).show()


## Filtering

- Use Spark boolean expressions with `filter` or `where`
- `df_sub = df.filter(F.col('salary') > 120000)`
- `df_f = df.filter(F.col('sex') == 'Female')`

### Operators
`>`, `>=`, `<`, `<=`, `==`, `!=`, `|` (or), `&` (and), `~` (not)

In [ ]:
df.filter(F.col('salary') > 120000).show()


# Basic statistics of the salary column

### Common functions
`count`, `sum`, `mean`, `mad`, `median`, `min`, `max`, `mode`, `abs`, `prod`, `std`, `var`, `sem`, `skew`, `kurt`, `quantile`, `cumsum`, `cumprod`, `cummax`, `cummin`

In [ ]:
df.agg(F.mean('salary').alias('avg_salary')).show()


In [ ]:
df.agg(F.stddev('salary').alias('std_salary')).show()


In [ ]:
df.agg(F.skewness('salary').alias('skew_salary')).show()


In [ ]:
df.agg(F.kurtosis('salary').alias('kurtosis_salary')).show()


In [ ]:
print(df.approxQuantile('salary', [0.5], 0.01)[0])


In [ ]:
df.select('salary').summary().show()


In [ ]:
print(df.select('salary').count())


In [ ]:
print(df.filter((F.col('sex') == 'Female') & (F.col('rank') == 'Prof') & (F.col('salary') > 120000)).count())


In [ ]:
print(df.filter((F.col('sex') == 'Female') & (F.col('rank') == 'Prof')).select('salary').count())


## Selection Methods

### Select columns
Select by name: `df.select('rank', 'sex', 'salary')`

### Select rows
Use `filter`, `where`, `limit`, `orderBy` for row subsets (Spark has no iloc/loc positional indexing).

In [ ]:
df.select('rank', 'service', 'salary').show(10)


In [ ]:
df_sub = df.filter(F.col('salary') > 120000)
df_sub.show()


In [ ]:
df_sub.select('rank', 'salary').show()


In [ ]:
df_sub.limit(20).select(df_sub.columns[:3]).show()


# Spark aggregate/reduce cell


In [ ]:
df.select(F.expr('coalesce(phd, 0) + coalesce(service, 0) + coalesce(salary, 0)').alias('row_sum')).show()


In [ ]:
df.select(*[F.mean(c).alias(c) for c in numeric_columns]).show()


In [ ]:
df.agg(F.stddev('salary').alias('salary_std')).show()


###  Standard scores: 

$ts\_stand = (df\_s - df\_s.mean()) / df\_s.std()$

$$
z = \frac{x - \mu}{\sigma} = \frac{\text{salary} - \text{mean}(\text{salary})}{\text{std}(\text{salary})} \quad \text{where } x = \text{salary},\ \mu = \text{mean},\ \sigma = \text{std}
$$

In [ ]:
stats = df.agg(F.mean('salary').alias('mean_salary'), F.stddev_pop('salary').alias('std_salary')).collect()[0]
df.select((F.col('salary') - F.lit(stats['mean_salary'])) / F.lit(stats['std_salary']).alias('std-salary')).show()


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkStandardScaler
assembler = VectorAssembler(inputCols=['salary'], outputCol='features')
assembled = assembler.transform(df.select('salary').na.drop())
scaler = SparkStandardScaler(inputCol='features', outputCol='scaled', withMean=True, withStd=True)
scaled_model = scaler.fit(assembled)
scaled_model.transform(assembled).select('scaled').show(5, truncate=False)


In [ ]:
stats = df.agg(F.mean('salary').alias('mean_salary'), F.stddev_pop('salary').alias('std_salary')).collect()[0]
df_app = df.withColumn('std-salary', (F.col('salary') - F.lit(stats['mean_salary'])) / F.lit(stats['std_salary']))
df_app.show()


### Other functions
`apply()`, `agg()` (e.g., `Df['salary'].agg(['sum','min'])`)

In [ ]:
df.agg(F.sum('salary').alias('sum_salary'), F.min('salary').alias('min_salary'), F.expr('aggregate(collect_list(salary), cast(1.0 as double), (acc, x) -> acc * x)').alias('prod_salary')).show()


In [ ]:
df.select(F.when(F.col('salary') < 90000, 'Low').otherwise('High').alias('salary_label')).show()


# Map

Series-only method that substitutes each value according to an input mapping. It’s optimized for element-wise value replacement or transformation.

## Map in DataFrame and Series

- `applymap()`: element-wise on DataFrame (removed since 2.1)
- `map()`: element-wise on Series

In [ ]:
mapping = F.create_map(F.lit('A'), F.lit('🟥'), F.lit('B'), F.lit('🟩'))
df.select(F.coalesce(F.element_at(mapping, F.col('discipline')), F.col('discipline')).alias('discipline_icon')).show()


In [ ]:
discipline_map = F.create_map(F.lit('A'), F.lit('🟥'), F.lit('B'), F.lit('🟩'))
sex_map = F.create_map(F.lit('Male'), F.lit('♂️'), F.lit('Female'), F.lit('♀️'))
df.select(F.coalesce(F.element_at(discipline_map, F.col('discipline')), F.col('discipline')).alias('discipline_icon'), F.coalesce(F.element_at(sex_map, F.col('sex')), F.col('sex')).alias('sex_icon')).show()


In [ ]:
df.select((F.col('salary') * 1.14).alias('salary_converted')).show()


In [ ]:
df.select((F.col('salary') * 1.14).alias('salary_converted')).show(5)

### str attribute
For string data in Series:
- `str.lower()`, `str.upper()`, `str.len()`, `str.split()`


In [ ]:
df.select(F.lower('sex').alias('sex_lower')).show()


## Sorting

- In Spark si usa `orderBy(...)` (o `sort(...)`)
- L'ordinamento e crescente di default

In [ ]:
df.orderBy('service').show()


In [ ]:
df.orderBy('discipline', 'service').show()


In [ ]:
df.orderBy(F.col('service').asc(), F.col('salary').desc()).show()


# Aggregate

## DataFrame groupby method

- Split data into groups based on criteria, returns a groupby object
- `groups` attribute: list group contents as a dict

Note: pass `sort=False` for speedup; from 2.1 need numeric_only

In [ ]:
df.groupBy('rank').count().show()

In [ ]:
df.groupBy('rank').agg(F.mean('salary').alias('mean_salary')).show()

In [ ]:
df.groupBy('rank').agg(F.mean('salary').alias('mean_salary')).show()

In [ ]:
df.groupBy('rank').count().show()

In [ ]:
numeric_cols = [c for c, d in df.dtypes if d in ('int', 'bigint', 'double', 'float')]
df.groupBy('rank').agg(*[F.mean(c).alias(c) for c in numeric_cols]).show()

# Row concatenate 
### concat function
Vertical or horizontal stacking.
- `axis`: 0 (index), 1 (columns)
- `join`: 'outer' (union, default), 'inner' (intersection)

In [ ]:
df1 = spark.createDataFrame([('AssocProf', 91786.23, 'A'), ('Prof', 123624.80, 'B'), ('AsstProf', 81362.78, 'A')], ['rank', 'salary', 'discipline'])
df1.show()


In [ ]:
df_u = df.unionByName(df1, allowMissingColumns=True)
df_u.show()


In [ ]:
df_u.show()


In [ ]:
print('Per affiancare colonne in Spark usa join su una chiave; non esiste concat axis=1.')

# table Merge

It works like a join, default is inner join

In [ ]:
df2 = spark.createDataFrame([('A', 'computer science'), ('B', 'math')], ['discipline', 'description'])
df1.join(df2, on='discipline', how='inner').show()


### merge function
High performance join operations.
- joining on common column or index
- `on`: columns to join on
- `how`: 'left', 'right', 'outer', 'inner' (default)
- `sort`: lexicographical order (default True)

### examples of concat method 

In [ ]:
data = [
    ('A', [10, 20, 30, 40, 50]),
    ('B', [15, 25, 35, 45, 55]),
    ('C', [12, 22, 32, 42, 52]),
]
data = spark.createDataFrame(data, ['Patient', 'measurements'])
data.show(truncate=False)

In [ ]:
max_len = data.select(F.max(F.size('measurements')).alias('max_len')).collect()[0]['max_len']
new_columns = data.select(*[F.try_element_at(F.col('measurements'), F.lit(i)).alias(f'measure_{i}') for i in range(1, max_len + 1)])
new_columns.show()


In [ ]:
expanded = data.select('Patient', F.posexplode_outer('measurements').alias('pos', 'value'))
wide = expanded.groupBy('Patient').pivot('pos').agg(F.first('value'))
for i, c in enumerate([c for c in wide.columns if c != 'Patient'], start=1):
    wide = wide.withColumnRenamed(c, f'measure_{i}')
wide.show()

### lists of various lengths

In [ ]:
data = [
    ('A', [10, 20, 30, 40, 50]),
    ('B', [15, 25]),
    ('C', [12, 22, 32]),
    ('D', [8]),
    ('E', []),
]
data = spark.createDataFrame(data, ['Patient', 'measurements'])
expanded = data.select('Patient', F.posexplode_outer('measurements').alias('pos', 'value'))
new_columns = expanded.groupBy('Patient').pivot('pos').agg(F.first('value'))
for i, c in enumerate([c for c in new_columns.columns if c != 'Patient'], start=1):
    new_columns = new_columns.withColumnRenamed(c, f'measure_{i}')
new_columns.show()

In [ ]:
new_columns.show()

### merge vs concat

```mermaid
graph TD
    subgraph "concat (axis=0)"
        A1[Box 1]
        A2[Box 2]
        A1 --> A3[Combined Vertically]
        A2 --> A3
    end
    subgraph "concat (axis=1)"
        B1[Box 1]
        B2[Box 2]
        B1 --- B3[Combined Horizontally]
        B2 --- B3
    end
```

# Body fat dataset (Spark)

## Dataset da Kaggle aggiornato su GitHub

Le variabili, da sinistra a destra, sono:

- Density da pesata idrostatica
- BodyFat percentuale (equazione di Siri 1956)
- Age (anni)
- Weight (lbs)
- Height (inches)
- Neck, Chest, Abdomen, Hip, Thigh, Knee, Ankle, Biceps, Forearm, Wrist (cm)

## Exercise

### 1) Proprieta base del dataset
- numero righe, colonne, celle

### 2) Verifica tipi colonna

### 3) Statistiche descrittive
- cercare valori anomali (es. Height)

### 4) Media Density
- per BodyFat > 0.25
- per BodyFat < 0.2

### 5) Aggiungi colonna BMI
$BMI = rac{	ext{Weight (kg)}}{	ext{Height (m)}^2}$

- Weight in kg: $1 	ext{ lb} = 0.453 	ext{ kg}$
- Height in m: $1 	ext{ inch} = 0.0254 	ext{ m}$

### 6) Classi categoriche BMI
- underweight, normal, overweight
- groupBy su cBMI e statistiche medie

In [ ]:
df = read_csv_any('https://raw.githubusercontent.com/s0SimoneP0s/dataset/refs/heads/main/bodyfat.csv')
df.show(5)

In [ ]:
# dataset basic properties and check data types
print((df.count(), len(df.columns)))
df.printSchema()

In [ ]:
df.describe().show()

In [ ]:
# cast cm from inches for Height field and kg from pounds for Weight
df = (df
    .withColumn('Height', F.col('Height') * F.lit(2.54))
    .withColumn('Weight', F.col('Weight') * F.lit(0.45))
)
df.describe().show()

In [ ]:
# check outliers with IQR rule in Spark
numeric = ['Density','BodyFat','Age','Weight','Height','Neck','Chest','Abdomen','Hip','Thigh','Knee','Ankle','Biceps','Forearm','Wrist']
exprs = []
for c in numeric:
    q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    exprs.append(F.when((F.col(c) < low) | (F.col(c) > high), 1).otherwise(0))

df_app = df.withColumn('outliers_score', sum(exprs))
df_app.select('outliers_score', *numeric[:4]).show(10)

In [ ]:
# scala outliers_score tra 0 e 1 in Spark
stats = df_app.agg(F.min('outliers_score').alias('mn'), F.max('outliers_score').alias('mx')).collect()[0]
mn, mx = stats['mn'], stats['mx']

if mx == mn:
    df_app = df_app.withColumn('outliers_score_norm', F.lit(0.0))
else:
    df_app = df_app.withColumn('outliers_score_norm', (F.col('outliers_score') - F.lit(mn)) / F.lit(mx - mn))

df_app.select('outliers_score', 'outliers_score_norm').show(10)

In [ ]:
# average Density for patients with BodyFat > 0.25
df.filter(F.col('BodyFat') > 0.25).agg(F.mean('Density').alias('avg_density')).collect()[0]['avg_density']

In [ ]:
# average Density for patients with BodyFat < 0.2
df.filter(F.col('BodyFat') < 0.2).agg(F.mean('Density').alias('avg_density')).collect()[0]['avg_density']

Add a BMI column whith formula: $BMI = \frac{\text{Weight (kg)}}{\text{Height (m)}^2}$

In [ ]:
# add BMI column with metric conversion
df = (df
    .withColumn('Weight_kg', F.col('Weight') * F.lit(0.45359237))
    .withColumn('Height_m', F.col('Height') * F.lit(0.0254))
    .withColumn('BMI', F.col('Weight_kg') / (F.col('Height_m') ** 2))
 )
df.select('Weight', 'Height', 'Weight_kg', 'Height_m', 'BMI').show(10)

In [ ]:
# categorical BMI

df = (df
    .withColumn(
        'cBMI',
        F.when(F.col('BMI') < 20, 'underweight')
         .when(F.col('BMI') < 27, 'normal')
         .otherwise('overweight')
    )
)
df.select('BMI', 'cBMI').show(10)

In [ ]:
# groupby cBMI and calc stat

df_app = df.drop('Weight', 'Height', 'Height_m', 'BMI')
df_app.groupBy('cBMI').agg(F.avg('Density').alias('avg_density'), F.avg('BodyFat').alias('avg_bodyfat')).show()

# **Tidy Data Features**
Three interrelated features make a dataset tidy:
- 1. Each variable is a column; each column is a variable.
- 2. Each observation is a row; each row is an observation.
- 3. Each value is a cell; each cell is a single value.


```mermaid
graph LR
  A[Variable] --> B[Column]
  C[Observation] --> D[Row]
  E[Value] --> F[Cell]
```

**The unpivot operation in Spark**
- Equivalent to melt: convert wide format to long format.
- Spark can use `stack(...)` SQL expression or `union` patterns.

**Example signature (stack-based)**
```python
df.selectExpr("id", "stack(3, 'A', A, 'B', B, 'C', C) as (variable, value)")
```

In [ ]:
df_messy = spark.createDataFrame([
    ('John Smith', None, 2, None, None),
    ('Jane Doe', 16, 11, 4, 1),
    ('Mary Johnson', 3, 1, None, 2),
], ['Name', 'Treatment A', 'Treatment B', 'Treatment C', 'Treatment D'])
df_messy.show()

Became "Tidy" (Long Format) with Spark `stack`/unpivot operations

In [ ]:
df_tidy = df_messy.selectExpr(
    'Name',
    "stack(4, 'Treatment A', `Treatment A`, 'Treatment B', `Treatment B`, 'Treatment C', `Treatment C`, 'Treatment D', `Treatment D`) as (Treatment, Result)"
)
df_tidy.show()

## PART 2 - Tidy Data dataset
- check the content of the file
- Melt the data to get a tidy dataset

In [ ]:
df = read_csv_any('https://raw.githubusercontent.com/s0SimoneP0s/dataset/refs/heads/main/pew-raw.csv')
df.show(5)

In [ ]:
df.columns

In [ ]:
# Unpivot to get a tidy DataFrame
value_cols = [' <$10k', ' $10-20k', '$20-30k', '$30-40k', ' $40-50k', '$50-75k']
stack_items = ', '.join([f"'{c}', `{c}`" for c in value_cols])
df_tidy = df.selectExpr('religion', f"stack({len(value_cols)}, {stack_items}) as (reddit, count)")
df_tidy.show()

### Tidy Data 2: product sales data

- Download the dataset and import it in your drive folder https://www.kaggle.com/datasets/ksabishek/product-sales-data

- Inspect Spark DataFrame content and validate data types with `printSchema()`. For dates use `to_date`/`to_timestamp`.

- Unpivot the data to obtain a tidy dataset

In [ ]:
spark = create_spark_session("Stats final csv")
df = read_csv_any('https://raw.githubusercontent.com/s0SimoneP0s/dataset/refs/heads/main/statsfinal.csv')
df = df.withColumnRenamed('Unnamed: 0', 'row_id')
df.show(5)
#if 'row_id' not in df.columns:
#    if '_c0' in df.columns:
#        df = df.withColumnRenamed('_c0','row_id')
#    elif 'Unnamed: 0' in df.columns:
#        df = df.withColumnRenamed('Unnamed: 0','row_id')
#    else:
#        df = df.withColumn('row_id', F.monotonically_increasing_id())

+------+----------+----+----+----+----+--------+--------+--------+--------+
|row_id|      Date|Q-P1|Q-P2|Q-P3|Q-P4|    S-P1|    S-P2|    S-P3|    S-P4|
+------+----------+----+----+----+----+--------+--------+--------+--------+
|     0|13-06-2010|5422|3725| 576| 907|17187.74| 23616.5| 3121.92| 6466.91|
|     1|14-06-2010|7047| 779|3578|1574|22338.99| 4938.86|19392.76|11222.62|
|     2|15-06-2010|1572|2082| 595|1145| 4983.24|13199.88|  3224.9| 8163.85|
|     3|16-06-2010|5657|2399|3140|1672|17932.69|15209.66| 17018.8|11921.36|
|     4|17-06-2010|3668|3207|2184| 708|11627.56|20332.38|11837.28| 5048.04|
+------+----------+----+----+----+----+--------+--------+--------+--------+
only showing top 5 rows


26/09/07 22:55:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
 Schema: _c0, Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


## dataset explain

- The products are P1, P2, P3 and P4.

- Q1- Total unit sales of product 1 and so... Q2 Q3 Q4

- S1- Total revenue from product 1 and so... S2 S3 S4

- The Unnamed is the transaction ID


In [29]:
# fix date while preserving original value
df = df.withColumn('Date_original', F.col('Date'))
df = df.withColumn('Date', F.expr("try_to_date(Date_original, 'dd-MM-yyyy')"))
df = df.withColumn('Bad_date', F.when(F.col('Date').isNull() & F.col('Date_original').isNotNull(), F.col('Date_original')).otherwise(F.lit(None)))
df = df.drop('Date_original')
df.show(5)

+------+----------+----+----+----+----+--------+--------+--------+--------+--------+
|row_id|      Date|Q-P1|Q-P2|Q-P3|Q-P4|    S-P1|    S-P2|    S-P3|    S-P4|Bad_date|
+------+----------+----+----+----+----+--------+--------+--------+--------+--------+
|     0|2010-06-13|5422|3725| 576| 907|17187.74| 23616.5| 3121.92| 6466.91|    NULL|
|     1|2010-06-14|7047| 779|3578|1574|22338.99| 4938.86|19392.76|11222.62|    NULL|
|     2|2010-06-15|1572|2082| 595|1145| 4983.24|13199.88|  3224.9| 8163.85|    NULL|
|     3|2010-06-16|5657|2399|3140|1672|17932.69|15209.66| 17018.8|11921.36|    NULL|
|     4|2010-06-17|3668|3207|2184| 708|11627.56|20332.38|11837.28| 5048.04|    NULL|
+------+----------+----+----+----+----+--------+--------+--------+--------+--------+
only showing top 5 rows


26/09/07 22:55:23 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
 Schema: _c0, Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [30]:
# robust parsing for residual invalid dates
parts = F.split(F.col('Bad_date'), '-')
df = df.withColumn('d', parts.getItem(0).cast('int')).withColumn('m', parts.getItem(1).cast('int')).withColumn('y', parts.getItem(2).cast('int'))
df = df.withColumn('safe_day', F.when(F.col('d') > 28, 28).otherwise(F.col('d')))
df = df.withColumn('Date_fixed_str', F.concat_ws('-', F.col('safe_day'), F.col('m'), F.col('y')))
df = df.withColumn('Date_fixed', F.expr("try_to_date(Date_fixed_str, 'd-M-yyyy')"))
df.show(5)

+------+----------+----+----+----+----+--------+--------+--------+--------+--------+----+----+----+--------+--------------+----------+
|row_id|      Date|Q-P1|Q-P2|Q-P3|Q-P4|    S-P1|    S-P2|    S-P3|    S-P4|Bad_date|   d|   m|   y|safe_day|Date_fixed_str|Date_fixed|
+------+----------+----+----+----+----+--------+--------+--------+--------+--------+----+----+----+--------+--------------+----------+
|     0|2010-06-13|5422|3725| 576| 907|17187.74| 23616.5| 3121.92| 6466.91|    NULL|NULL|NULL|NULL|    NULL|              |      NULL|
|     1|2010-06-14|7047| 779|3578|1574|22338.99| 4938.86|19392.76|11222.62|    NULL|NULL|NULL|NULL|    NULL|              |      NULL|
|     2|2010-06-15|1572|2082| 595|1145| 4983.24|13199.88|  3224.9| 8163.85|    NULL|NULL|NULL|NULL|    NULL|              |      NULL|
|     3|2010-06-16|5657|2399|3140|1672|17932.69|15209.66| 17018.8|11921.36|    NULL|NULL|NULL|NULL|    NULL|              |      NULL|
|     4|2010-06-17|3668|3207|2184| 708|11627.56|20332.3

26/09/07 22:55:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
 Schema: _c0, Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [31]:
df = df.withColumn('Date_fixed', F.coalesce(F.col('Date'), F.col('Date_fixed')))
df.filter(F.col('Date').isNull()).show(5)

+------+----+----+----+----+----+--------+--------+--------+--------+----------+---+---+----+--------+--------------+----------+
|row_id|Date|Q-P1|Q-P2|Q-P3|Q-P4|    S-P1|    S-P2|    S-P3|    S-P4|  Bad_date|  d|  m|   y|safe_day|Date_fixed_str|Date_fixed|
+------+----+----+----+----+----+--------+--------+--------+--------+----------+---+---+----+--------+--------------+----------+
|   109|NULL|4986| 342|4978| 558|15805.62| 2168.28|26980.76| 3978.54| 31-9-2010| 31|  9|2010|      28|     28-9-2010|2010-09-28|
|   170|NULL|4632|3930| 523|1581|14683.44| 24916.2| 2834.66|11272.53|31-11-2010| 31| 11|2010|      28|    28-11-2010|2010-11-28|
|   473|NULL|2242| 401|5926| 789| 7107.14| 2542.34|32118.92| 5625.57| 31-9-2011| 31|  9|2011|      28|     28-9-2011|2011-09-28|
|   534|NULL| 325|3476|4588|1771| 1030.25|22037.84|24866.96|12627.23|31-11-2011| 31| 11|2011|      28|    28-11-2011|2011-11-28|
|   836|NULL|1003| 256|1346|1449| 3179.51| 1623.04| 7295.32|10331.37| 31-9-2012| 31|  9|2012|    

26/09/07 22:55:26 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
 Schema: _c0, Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [32]:
df = df.withColumn('Date', F.coalesce(F.col('Date_fixed'), F.col('Date')))
df.filter(F.col('Date').isNull()).show(5)

+------+----+----+----+----+----+----+----+----+----+--------+---+---+---+--------+--------------+----------+
|row_id|Date|Q-P1|Q-P2|Q-P3|Q-P4|S-P1|S-P2|S-P3|S-P4|Bad_date|  d|  m|  y|safe_day|Date_fixed_str|Date_fixed|
+------+----+----+----+----+----+----+----+----+----+--------+---+---+---+--------+--------------+----------+
+------+----+----+----+----+----+----+----+----+----+--------+---+---+---+--------+--------------+----------+



26/09/07 22:55:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
 Schema: _c0, Date, Q-P1, Q-P2, Q-P3, Q-P4, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [33]:
measurements_df = spark.createDataFrame([
    ('A', [10, 20, 30, 40, 50]),
    ('B', [15, 25]),
    ('C', [12, 22, 32]),
    ('D', [8]),
    ('E', []),
], ['Patient', 'measurements'])
max_len = measurements_df.select(F.max(F.size('measurements')).alias('max_len')).collect()[0]['max_len']
new_columns = measurements_df.select(*[F.try_element_at(F.col('measurements'), F.lit(i)).alias(f'measure_{i}') for i in range(1, max_len + 1)])
new_columns.show()

+---------+---------+---------+---------+---------+
|measure_1|measure_2|measure_3|measure_4|measure_5|
+---------+---------+---------+---------+---------+
|       10|       20|       30|       40|       50|
|       15|       25|     NULL|     NULL|     NULL|
|       12|       22|       32|     NULL|     NULL|
|        8|     NULL|     NULL|     NULL|     NULL|
|     NULL|     NULL|     NULL|     NULL|     NULL|
+---------+---------+---------+---------+---------+



In [34]:
df.columns

['row_id',
 'Date',
 'Q-P1',
 'Q-P2',
 'Q-P3',
 'Q-P4',
 'S-P1',
 'S-P2',
 'S-P3',
 'S-P4',
 'Bad_date',
 'd',
 'm',
 'y',
 'safe_day',
 'Date_fixed_str',
 'Date_fixed']

In [35]:
print('Spark DataFrames do not have an index')

Spark DataFrames do not have an index


In [36]:
# Unpivot Quantity columns
q_cols = [c for c in df.columns if c.startswith('Q-')]
q_items = ', '.join([f"'{c}', `{c}`" for c in q_cols])
df_q = df.selectExpr('row_id', f"stack({len(q_cols)}, {q_items}) as (Product, Quantity)")
df_q = df_q.withColumn('Product', F.regexp_replace('Product', '^Q-', ''))
df_q.show(5)

+------+-------+--------+
|row_id|Product|Quantity|
+------+-------+--------+
|     0|     P1|    5422|
|     0|     P2|    3725|
|     0|     P3|     576|
|     0|     P4|     907|
|     1|     P1|    7047|
+------+-------+--------+
only showing top 5 rows


26/09/07 22:55:37 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Q-P1, Q-P2, Q-P3, Q-P4
 Schema: _c0, Q-P1, Q-P2, Q-P3, Q-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [37]:
# Unpivot Revenue columns
s_cols = [c for c in df.columns if c.startswith('S-')]
s_items = ', '.join([f"'{c}', `{c}`" for c in s_cols])
df_s = df.selectExpr('row_id', f"stack({len(s_cols)}, {s_items}) as (Product, Revenue)")
df_s = df_s.withColumn('Product', F.regexp_replace('Product', '^S-', ''))
df_s.show(5)

+------+-------+--------+
|row_id|Product| Revenue|
+------+-------+--------+
|     0|     P1|17187.74|
|     0|     P2| 23616.5|
|     0|     P3| 3121.92|
|     0|     P4| 6466.91|
|     1|     P1|22338.99|
+------+-------+--------+
only showing top 5 rows


26/09/07 22:55:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , S-P1, S-P2, S-P3, S-P4
 Schema: _c0, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [38]:
df_tidy = df_q.join(df_s, on=['row_id', 'Product'], how='inner')
df_tidy.show(5)

26/09/07 22:55:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Q-P1, Q-P2, Q-P3, Q-P4
 Schema: _c0, Q-P1, Q-P2, Q-P3, Q-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


+------+-------+--------+--------+
|row_id|Product|Quantity| Revenue|
+------+-------+--------+--------+
|     0|     P1|    5422|17187.74|
|     0|     P2|    3725| 23616.5|
|     0|     P3|     576| 3121.92|
|     0|     P4|     907| 6466.91|
|     1|     P1|    7047|22338.99|
+------+-------+--------+--------+
only showing top 5 rows


26/09/07 22:55:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , S-P1, S-P2, S-P3, S-P4
 Schema: _c0, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [39]:
df_date = df.select('row_id', 'Date')
df_date.show(5)

+------+----------+
|row_id|      Date|
+------+----------+
|     0|2010-06-13|
|     1|2010-06-14|
|     2|2010-06-15|
|     3|2010-06-16|
|     4|2010-06-17|
+------+----------+
only showing top 5 rows


26/09/07 22:55:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date
 Schema: _c0, Date
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [40]:
df_final = df_tidy.join(df.select('row_id', 'Date'), on='row_id', how='left')
df_final = df_final.drop('row_id')
df_final.show(5)

26/09/07 22:55:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Q-P1, Q-P2, Q-P3, Q-P4
 Schema: _c0, Q-P1, Q-P2, Q-P3, Q-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv
26/09/07 22:55:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date
 Schema: _c0, Date
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


+-------+--------+--------+----------+
|Product|Quantity| Revenue|      Date|
+-------+--------+--------+----------+
|     P1|    5422|17187.74|2010-06-13|
|     P2|    3725| 23616.5|2010-06-13|
|     P3|     576| 3121.92|2010-06-13|
|     P4|     907| 6466.91|2010-06-13|
|     P1|    7047|22338.99|2010-06-14|
+-------+--------+--------+----------+
only showing top 5 rows


26/09/07 22:55:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , S-P1, S-P2, S-P3, S-P4
 Schema: _c0, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [41]:
df_final.printSchema()
print((df_final.count(), len(df_final.columns)))

root
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Date: date (nullable = true)



26/09/07 22:55:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv
26/09/07 22:55:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Q-P1, Q-P2, Q-P3, Q-P4
 Schema: _c0, Q-P1, Q-P2, Q-P3, Q-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


(18400, 4)


26/09/07 22:55:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , S-P1, S-P2, S-P3, S-P4
 Schema: _c0, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


In [42]:
df_final_app = df_final.drop('Date')
df_final_app.groupBy('Product').agg(F.avg('Quantity').alias('avg_quantity'), F.avg('Revenue').alias('avg_revenue')).show()

26/09/07 22:55:48 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Q-P1, Q-P2, Q-P3, Q-P4
 Schema: _c0, Q-P1, Q-P2, Q-P3, Q-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv
26/09/07 22:55:48 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv
26/09/07 22:55:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , S-P1, S-P2, S-P3, S-P4
 Schema: _c0, S-P1, S-P2, S-P3, S-P4
Expected: _c0 but found: 
CSV file: file:///media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/exam/BD/experiments/03-engine_spark/esercizi/.spark-tmp/statsfinal.csv


+-------+------------------+------------------+
|Product|      avg_quantity|       avg_revenue|
+-------+------------------+------------------+
|     P2|2130.2815217391303|13505.984847826052|
|     P3|           3145.74| 17049.91079999996|
|     P4|            1123.5| 8010.555000000009|
|     P1| 4121.849130434783| 13066.26174347828|
+-------+------------------+------------------+



# exercise
### Hands on SALARIES (slide 53)

*  Given the dataset df_2 defined below, merge df and df_2 keeping record union (outer).

*  How many nans in the dataset?

*  Drop rows containing nan values

*  Sort values according to discipline and Dept

*  Compute the average distribution of Females anf Males in the three Departments.

In [ ]:
spark = create_spark_session("Salaries csv")
base = read_csv_any('https://raw.githubusercontent.com/s0SimoneP0s/dataset/refs/heads/main/Salaries.csv')
import random

r = random.randint(20, 70)
df_2 = (base
    .select('rank', 'phd', 'salary')
    .orderBy(F.rand(seed=42))
    .limit(r)
    .withColumn('Dept', F.when(F.rand(seed=1) < 0.33, 'Math').when(F.rand(seed=2) < 0.66, 'CS').otherwise('Phys'))
)
df_2.show(5)

26/09/07 22:57:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+--------+---+------+----+
|    rank|phd|salary|Dept|
+--------+---+------+----+
|AsstProf|  8| 75044|  CS|
|    Prof| 36|144651|  CS|
|AsstProf|  7| 63100|Math|
|    Prof| 24|161101|Math|
|    Prof| 18|126300|  CS|
+--------+---+------+----+
only showing top 5 rows


In [51]:
# merge outer
df_merge = df.join(df_2, on=['rank', 'phd', 'salary'], how='outer')
df_merge.show(5)

AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `rank` cannot be resolved on the left side of the join. The left-side columns: [`Bad_date`, `Date`, `Date_fixed`, `Date_fixed_str`, `Q-P1`, `Q-P2`, `Q-P3`, `Q-P4`, `S-P1`, `S-P2`, `S-P3`, `S-P4`, `d`, `m`, `row_id`, `safe_day`, `y`]. SQLSTATE: 42703

In [45]:
# nan count in the dataset
null_totals = df_merge.select(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_merge.columns]).collect()[0]
print(sum(null_totals)) # == isnull()

NameError: name 'df_merge' is not defined

In [46]:
# row drop nan and check
df_merge.dropna().count()

NameError: name 'df_merge' is not defined

In [47]:
# save result
df_clear = df_merge.dropna()

NameError: name 'df_merge' is not defined

In [48]:
# Sort values according to discipline and Dept
df_clear.orderBy('discipline', 'Dept').show(5)

NameError: name 'df_clear' is not defined

In [49]:
# Compute average distribution of Females and Males in the three Departments
w = Window.partitionBy('Dept')
res = (df_clear.groupBy('Dept', 'sex').count()
       .withColumn('dept_total', F.sum('count').over(w))
       .withColumn('ratio', F.col('count') / F.col('dept_total')))
res.show()

NameError: name 'df_clear' is not defined

## Tidy Data dataset
### Wide-to-long in Spark

Spark non ha una funzione dedicata wide_to_long: si usa unpivot con `stack(...)` oppure `selectExpr` + `union`.
| Step | Wide-to-long in Spark |
| :--- | :--- |
| 1. | Individua le colonne chiave (`id_vars`). |
| 2. | Individua i gruppi di colonne wide (`stubnames`). |
| 3. | Usa `stack` per convertire colonne in righe. |
| 4. | Estrai il suffisso (es. anno/prodotto) con regex. |
| 5. | Esegui join tra i gruppi long sulle chiavi. |

In [ ]:
df = spark.createDataFrame([
    (0, 'a', 'd', 2.5, 3.2, 0.1),
    (1, 'b', 'e', 1.2, 1.3, -0.2),
    (2, 'c', 'f', 0.7, 0.1, 1.5),
], ['id', 'A1970', 'A1980', 'B1970', 'B1980', 'X'])
df.show()

In [ ]:
# wide-to-long transformation
long_A = df.selectExpr('id', 'X', "stack(2, '1970', A1970, '1980', A1980) as (renamed, A)")
long_B = df.selectExpr('id', 'X', "stack(2, '1970', B1970, '1980', B1980) as (renamed, B)")
long_A.join(long_B, on=['id', 'X', 'renamed'], how='inner').show()

In [ ]:
long_A = df.selectExpr('id', 'X', "stack(2, '1970', A1970, '1980', A1980) as (var, A)")
long_B = df.selectExpr('id', 'X', "stack(2, '1970', B1970, '1980', B1980) as (var, B)")
long_A.join(long_B, on=['id', 'X', 'var'], how='inner').show()

### Exercise:
Use the dataset from the previous (sales) exercise and make it tidy using the wide-to-long method

In [ ]:
df = read_csv_any('https://raw.githubusercontent.com/s0SimoneP0s/dataset/refs/heads/main/statsfinal.csv')
df = df.withColumnRenamed('Unnamed: 0', 'row_id')
df.show(5)

In [ ]:
q_cols = [c for c in df.columns if c.startswith('Q-')]
s_cols = [c for c in df.columns if c.startswith('S-')]
q_stack = ', '.join([f"'{c[2:]}', `{c}`" for c in q_cols])
s_stack = ', '.join([f"'{c[2:]}', `{c}`" for c in s_cols])
q_long = df.selectExpr('_c0 as row_id', 'Date', f"stack({len(q_cols)}, {q_stack}) as (Product, Q)")
s_long = df.selectExpr('_c0 as row_id', 'Date', f"stack({len(s_cols)}, {s_stack}) as (Product, S)")
q_long.join(s_long, on=['row_id', 'Date', 'Product'], how='inner').show(5)

## Stack and Unstack in Spark

- Spark non ha `DataFrame.stack/unstack` come API indice-oriented.
- Per wide -> long usa `stack(...)` in `selectExpr`.
- Per long -> wide usa `groupBy(...).pivot(...).agg(...)`.

In [ ]:
df = spark.createDataFrame([
    ('cat', 0, 1),
    ('dog', 2, 3),
], ['animal', 'weight', 'height'])
df.show()

In [ ]:
# Stacked Data (wide -> long)
df_app = df.selectExpr('animal', "stack(2, 'weight', weight, 'height', height) as (feature, value)")
df_app.show()

In [ ]:
# Unstacked Data (long -> wide)
df_app.groupBy('animal').pivot('feature').agg(F.first('value')).show()

In [ ]:
# Pivot con diversa disposizione
df_app.groupBy('feature').pivot('animal').agg(F.first('value')).show()

In [ ]:
df = spark.createDataFrame([
    ('North', 100, 120, 130),
    ('South', 150, 160, 170),
    ('East', 200, 220, 240),
    ('West', 250, 280, 300),
], ['Region', 'Product_A', 'Product_B', 'Product_C'])
df.show()

In [ ]:
# Wide -> long
(df.selectExpr(
    'Region',
    "stack(3, 'Product_A', Product_A, 'Product_B', Product_B, 'Product_C', Product_C) as (Product, Value)"
).show())

#### Further examples using map, stack and **explode**

In [ ]:
# Example DataFrame with a column containing lists of various lengths

df = spark.createDataFrame([
    ('A', [10, 20, 30, 40, 50]),
    ('B', [15, 25]),
    ('C', [12, 22, 32]),
    ('D', [8]),
    ('E', []),
], ['Patient', 'measurements'])

In [ ]:
# explode per espandere le liste in righe
newco = df.select('Patient', F.posexplode_outer('measurements').alias('idx', 'value'))
newco.show()

In [ ]:
# ricostruzione in formato wide
new_da = newco.groupBy('Patient').pivot('idx').agg(F.first('value'))
new_da.show()

In [ ]:
# usando stack su colonne create in precedenza
new_da_long = new_da.selectExpr('Patient', "stack(5, '0', `0`, '1', `1`, '2', `2`, '3', `3`, '4', `4`) as (measure_id, measure_value)")
new_da_long.show()

In [ ]:
# explodes the 'measurements' column into rows
df.select('Patient', F.explode_outer('measurements').alias('measurement')).show()

## Pivoting

- `pivot`, command reshapes a DataFrame from a long format to a wide format by pivoting the values of a column into new columns.

**`DataFrame.pivot(columns, index, values)`** 
- **columns**: Column to use to make new frame’s columns. 
- **index**: Column to use to make new frame’s index. defaults to current index. 
- **values**: Column(s) to use for populating new DF values. If not specified, all remaining columns will be used 

In [ ]:
data = spark.range(0, 20).withColumn('value', F.rand(seed=7)).withColumn('variable', F.when((F.col('id') % 2) == 0, 'A').otherwise('B')).withColumn('category', F.expr("element_at(array('type1','type2','type3','type4'), int(rand(11)*4)+1)")).withColumn('date', F.expr("add_months(to_date('2000-01-01'), int(id/2))")).select('date', 'variable', 'category', 'value')
data.orderBy('date').show(5)

In [ ]:
# single column pivoted
data.groupBy('date').pivot('variable').agg(F.first('value')).show(5)

In [ ]:
# single two column pivoted
data.groupBy('date', 'category').pivot('variable').agg(F.first('value')).show(5)

### Exercise:  
Use pivoting to get the the results shown in slide 27

In [ ]:
df = spark.createDataFrame([
    ('North', 100, 120, 130),
    ('South', 150, 160, 170),
    ('East', 200, 220, 240),
    ('West', 250, 280, 300),
], ['Region', 'Product_A', 'Product_B', 'Product_C'])
df.show()

In [ ]:
df_app = df.selectExpr(
    'Region',
    "stack(3, 'A', Product_A, 'B', Product_B, 'C', Product_C) as (Product_code, Product)"
)
df_app.show()

In [ ]:
# alternativa long format
df_app = df.selectExpr('Region', "stack(3, 'Product_A', Product_A, 'Product_B', Product_B, 'Product_C', Product_C) as (Product, Value)")
df_app.show()

In [ ]:
# single index
df_app.select("Region","Product").show()

In [ ]:
# Wide Format Data recovery
df_app.groupBy('Region').pivot('Product').agg(F.first('Value')).show()

# plotting data

## scatter

In [ ]:
import matplotlib.pyplot as plt

data = [('USA', 45000), ('Canada', 42000), ('Germany', 52000), ('UK', 49000), ('France', 47000)]
df = spark.createDataFrame(data, ['Country', 'GDP'])
rows = df.orderBy('Country').collect()
xs = [r['Country'] for r in rows]
ys = [r['GDP'] for r in rows]

plt.figure(figsize=(5, 5))
plt.scatter(xs, ys)
plt.xlabel('Country')
plt.ylabel('GDP')
plt.show()

# line

In [ ]:
import matplotlib.pyplot as plt

data = [('USA', 45000), ('Canada', 42000), ('Germany', 52000), ('UK', 49000), ('France', 47000)]
df = spark.createDataFrame(data, ['Country', 'GDP'])
rows = df.orderBy('Country').collect()
xs = [r['Country'] for r in rows]
ys = [r['GDP'] for r in rows]

plt.figure(figsize=(5, 5))
plt.plot(xs, ys, marker='o')
plt.xlabel('Country')
plt.ylabel('GDP')
plt.show()

## pie

In [ ]:
import matplotlib.pyplot as plt

data = [('Tasks Pending', 300), ('Tasks Ongoing', 500), ('Tasks Completed', 700)]
df = spark.createDataFrame(data, ['Label', 'Tasks'])
rows = df.collect()
labels = [r['Label'] for r in rows]
values = [r['Tasks'] for r in rows]

plt.figure(figsize=(5, 5))
plt.pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
plt.show()

## Groupby

### PETS

In [ ]:
import matplotlib.pyplot as plt

pets = spark.createDataFrame([
    ('john', 23, 'M', 'california', 2, 5),
    ('mary', 78, 'F', 'dc', 0, 1),
    ('peter', 22, 'M', 'california', 0, 0),
    ('jeff', 19, 'M', 'dc', 3, 5),
    ('bill', 45, 'M', 'california', 2, 2),
    ('lisa', 33, 'F', 'texas', 1, 2),
    ('jose', 20, 'M', 'texas', 4, 3),
], ['name', 'age', 'gender', 'state', 'num_children', 'num_pets'])

pets.show(2)

print('Scatter')
pts = pets.select('num_children', 'num_pets').collect()
plt.scatter([r['num_children'] for r in pts], [r['num_pets'] for r in pts], color='purple')
plt.xlabel('num_children')
plt.ylabel('num_pets')
plt.show()

print('Lines children | pets')
rows = pets.orderBy('name').select('name', 'num_children', 'num_pets').collect()
names = [r['name'] for r in rows]
children = [r['num_children'] for r in rows]
petn = [r['num_pets'] for r in rows]
plt.plot(names, children, label='num_children')
plt.plot(names, petn, color='red', label='num_pets')
plt.legend()
plt.show()

print('Bar group by children')
ch = pets.groupBy('num_children').count().orderBy('num_children').collect()
plt.bar([r['num_children'] for r in ch], [r['count'] for r in ch])
plt.show()

print('Bar group by pets')
pp = pets.groupBy('num_pets').count().orderBy('num_pets').collect()
plt.bar([r['num_pets'] for r in pp], [r['count'] for r in pp])
plt.show()

print('Bar group by state')
st = pets.groupBy('state').count().orderBy('state').collect()
plt.bar([r['state'] for r in st], [r['count'] for r in st])
plt.show()